## 🔍 Questões da Análise

1. Quantos registros existem na base?
2. Quantas equipes únicas mandantes existem na base?
3. Quantas vezes as equipes mandantes saíram vitoriosas?
4. Quantas vezes as equipes visitantes saíram vitoriosas?
5. Quantas partidas resultaram em empate?
6. Quantas partidas foram realizadas em cada país?
7. Qual país teve mais partidas?
8. Qual a partida com maior número de gols?
9. Qual a maior goleada?
10. Quantos jogos ocorreram no Brasil?

## Passo 1: 
##### Para analisar a base de dados será criado uma temp view a partir do volume, visando construir colunas auxiliares que facilitem a analise. Lembrando de desconsiderar os jogos com neutral = true.

In [0]:
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    max as spark_max, min as spark_min, round as spark_round,
    current_timestamp, percentile_approx, stddev, countDistinct
)
from pyspark.sql import functions as F
from datetime import datetime
from pyspark.sql import Window as W

def build_query():
    try:
        query ="""
        SELECT *,
            case when home_scoreHome<away_scoreAway then 'away_win'
                 when home_scoreHome>away_scoreAway then 'home_win'
                 when home_scoreHome=away_scoreAway then 'draw'
            end as winner,
            home_scoreHome + away_scoreAway as total_goals,
            case when home_scoreHome<away_scoreAway then away_teamName
                 when home_scoreHome>away_scoreAway then home_teamName
                 when home_scoreHome=away_scoreAway then null
            end as winner_teamName
        FROM raw_data
    """
        print('Query construída com sucesso')
        return query
    except Exception as e:
        print(e)

def create_dataframe():
    volume_path='/Volumes/workspace/default/jogos_internacionais_resultados/results_11ABDR.csv'
    print('Lendo dados do volume')
    try:
        df_inicial=spark.read.csv(volume_path, header=True, inferSchema=True)
        print('Dados lidos com sucesso')
        print('Mostrando 10 linhas após a leitura')
        df_inicial.limit(10).display()
        print('Mostrando schema')
        df_inicial.printSchema()
        return df_inicial
    except Exception as e:
        print(e)
        df_inicial=None
        return df_inicial

def transform_data(dataframe, query):
    dataframe.createOrReplaceTempView("raw_data")
    query=build_query()
    df_transformado= spark.sql(query)
    print('Dados transformados com sucesso')
    df_transformado.createOrReplaceTempView("transformed_data")
    print('Mostrando 10 linhas após a Transformação')
    df_transformado.limit(10).display()
    print('Mostrando schema')
    df_transformado.printSchema()
    return df_transformado

def main():
    query=build_query()
    df_inicial=create_dataframe()
    df_transformado=transform_data(df_inicial, query)
    return df_transformado
# DBTITLE 1,Testes
df_transformado=main()




Query construída com sucesso
Lendo dados do volume
Dados lidos com sucesso
Mostrando 10 linhas após a leitura


date,home_teamName,away_teamName,home_scoreHome,away_scoreAway,tournamentName,cityCity,countryCountry,neutralTRUE
1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,false
1873-03-08,England,Scotland,4,2,Friendly,London,England,false
1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,false
1875-03-06,England,Scotland,2,2,Friendly,London,England,false
1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,false
1876-03-25,Scotland,Wales,4,0,Friendly,Glasgow,Scotland,false
1877-03-03,England,Scotland,1,3,Friendly,London,England,false
1877-03-05,Wales,Scotland,0,2,Friendly,Wrexham,Wales,false
1878-03-02,Scotland,England,7,2,Friendly,Glasgow,Scotland,false
1878-03-23,Scotland,Wales,9,0,Friendly,Glasgow,Scotland,false


Mostrando schema
root
 |-- date: date (nullable = true)
 |-- home_teamName: string (nullable = true)
 |-- away_teamName: string (nullable = true)
 |-- home_scoreHome: integer (nullable = true)
 |-- away_scoreAway: integer (nullable = true)
 |-- tournamentName: string (nullable = true)
 |-- cityCity: string (nullable = true)
 |-- countryCountry: string (nullable = true)
 |-- neutralTRUE: boolean (nullable = true)

Query construída com sucesso
Dados transformados com sucesso
Mostrando 10 linhas após a Transformação


date,home_teamName,away_teamName,home_scoreHome,away_scoreAway,tournamentName,cityCity,countryCountry,neutralTRUE,winner,total_goals,winner_teamName
1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,false,draw,0,null
1873-03-08,England,Scotland,4,2,Friendly,London,England,false,home_win,6,England
1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,false,home_win,3,Scotland
1875-03-06,England,Scotland,2,2,Friendly,London,England,false,draw,4,null
1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,false,home_win,3,Scotland
1876-03-25,Scotland,Wales,4,0,Friendly,Glasgow,Scotland,false,home_win,4,Scotland
1877-03-03,England,Scotland,1,3,Friendly,London,England,false,away_win,4,Scotland
1877-03-05,Wales,Scotland,0,2,Friendly,Wrexham,Wales,false,away_win,2,Scotland
1878-03-02,Scotland,England,7,2,Friendly,Glasgow,Scotland,false,home_win,9,Scotland
1878-03-23,Scotland,Wales,9,0,Friendly,Glasgow,Scotland,false,home_win,9,Scotland


Mostrando schema
root
 |-- date: date (nullable = true)
 |-- home_teamName: string (nullable = true)
 |-- away_teamName: string (nullable = true)
 |-- home_scoreHome: integer (nullable = true)
 |-- away_scoreAway: integer (nullable = true)
 |-- tournamentName: string (nullable = true)
 |-- cityCity: string (nullable = true)
 |-- countryCountry: string (nullable = true)
 |-- neutralTRUE: boolean (nullable = true)
 |-- winner: string (nullable = true)
 |-- total_goals: integer (nullable = true)
 |-- winner_teamName: string (nullable = true)



## Passo 2:
##### Após finalizar a criação das visões temporárias é chegada a hora de responder as perguntas realizadas:

### 1. Quantos registros existem na base?

In [0]:
qtd_registros_base=df_transformado.count()
qtd_validos=df_transformado.filter(df_transformado.neutralTRUE==False).count()
print(f'Quantidade de registros na base são {qtd_registros_base}, sendo que destes {qtd_validos} são válidos')

Quantidade de registros na base são 40839, sendo que destes 30680 são válidos


### 2. Quantas equipes únicas mandantes existem na base?

In [0]:
qtd_eq_uni_home=df_transformado.select('home_teamName').distinct().count()
qtd_validos_eq_uni_home=df_transformado.filter(df_transformado.neutralTRUE==False).select('home_teamName').distinct().count()
print(f'Quantidade de equipes únicas mandantes usando o campo home_teamName, são no total {qtd_eq_uni_home}, sendo que destes apenas {qtd_validos_eq_uni_home} mandantes tiveram jogos considerados válidos')

Quantidade de equipes únicas mandantes usando o campo home_teamName, são no total 309, sendo que destes apenas 267 mandantes tiveram jogos considerados válidos


### 3.Quantas vezes as equipes mandantes saíram vitoriosas?

In [0]:
qtd_eq_win_home=df_transformado.filter(df_transformado.winner=='home_win').count()
qtd_eq_win_home_validos=df_transformado.filter((df_transformado.neutralTRUE==False) & (df_transformado.winner=='home_win')).count()
print(f'Quantidade de vezes que a equipe mandante saiu como vencedora, tendo como base o campo winner, foram {qtd_eq_win_home}, no entanto se considerarmos apenas os vaálidos foram {qtd_eq_win_home_validos}')

Quantidade de vezes que a equipe mandante saiu como vencedora, tendo como base o campo winner, foram 19864, no entanto se considerarmos apenas os vaálidos foram 15481


### 4.Quantas vezes as equipes visitantes saíram vitoriosas?

In [0]:
qtd_eq_win_away=df_transformado.filter(df_transformado.winner=='away_win').count()
qtd_eq_win_away_validos=df_transformado.filter((df_transformado.neutralTRUE==False) & (df_transformado.winner=='away_win')).count()
print(f'Quantidade de vezes que a equipe visitante saiu como vencedora, tendo como base o campo winner, foram {qtd_eq_win_away}, no entanto se considerarmos apenas os vaálidos foram {qtd_eq_win_away_validos}')

Quantidade de vezes que a equipe visitante saiu como vencedora, tendo como base o campo winner, foram 11544, no entanto se considerarmos apenas os vaálidos foram 8086


### 5. Quantas partidas resultaram em empate?

In [0]:
qtd_eq_win_draw=df_transformado.filter(df_transformado.winner=='draw').count()
qtd_eq_win_draw_validos=df_transformado.filter((df_transformado.neutralTRUE==False) & (df_transformado.winner=='draw')).count()
print(f'Quantidade de vezes que houve empate, tendo como base o campo winner, foram {qtd_eq_win_draw}, no entanto se considerarmos apenas os vaálidos foram {qtd_eq_win_draw_validos}')

Quantidade de vezes que houve empate, tendo como base o campo winner, foram 9431, no entanto se considerarmos apenas os vaálidos foram 7113


### 6.Quantas partidas foram realizadas em cada país?

In [0]:
qtd_part_country= df_transformado.groupBy('countryCountry').count().orderBy('count', ascending=False)
print(f'Quantidade de partidas por país, ordenado por quantidade de partidas:')
display(qtd_part_country)

qtd_part_country_validos= df_transformado.filter(df_transformado.neutralTRUE==False).groupBy('countryCountry').count().orderBy('count', ascending=False)
print(f'Quantidade de partidas por país, ordenado por quantidade de partidas validas:')
display(qtd_part_country_validos)

Quantidade de partidas por país, ordenado por quantidade de partidas:


countryCountry,count
United States,1144
France,801
England,687
Malaysia,644
Sweden,637
Germany,581
Brazil,529
Spain,517
Thailand,483
Italy,480


Quantidade de partidas por país, ordenado por quantidade de partidas validas:


countryCountry,count
England,471
Sweden,469
France,448
Hungary,421
Germany,414
Spain,398
United States,392
Austria,389
Norway,383
Switzerland,382


### 7. Qual país teve mais partidas?

In [0]:
qtd_part_country_first= df_transformado.groupBy('countryCountry').count().orderBy('count', ascending=False)
nome_pais=qtd_part_country_first.first()['countryCountry']
quantidade_jogos=qtd_part_country_first.first()['count']
qtd_part_country_valido=df_transformado.filter(df_transformado.neutralTRUE==False).groupBy('countryCountry').count().orderBy('count', ascending=False)
nome_pais_valido=qtd_part_country_validos.first()['countryCountry']
quantidade_jogos_validos=qtd_part_country_validos.first()['count']
print(f'O país com mais partidas é o {nome_pais}, com {quantidade_jogos} de partidas realizadas. No entanto se considerarmos apenas os jogos válidos, o país com mais partidas é o {nome_pais_valido}, com {quantidade_jogos_validos} de partidas realizadas')

O país com mais partidas é o United States, com 1144 de partidas realizadas. No entanto se considerarmos apenas os jogos válidos, o país com mais partidas é o England, com 471 de partidas realizadas


### 8. Qual a partida com maior número de gols?

In [0]:
### Contagem sem considerar se é valido ou não:
qtd_max_goals= df_transformado.orderBy('total_goals', ascending=False).limit(1)
nome_pais_mandante=qtd_max_goals.first()['home_teamName']
nome_pais_visitante=qtd_max_goals.first()['away_teamName']
quantidade_gols=qtd_max_goals.first()['total_goals']
nome_torneio=qtd_max_goals.first()['tournamentName']

## Contagem considerando os validos:
qtd_max_goals_validos= df_transformado.filter(df_transformado.neutralTRUE==False).orderBy('total_goals', ascending=False).limit(1)
nome_pais_mandante_valido=qtd_max_goals_validos.first()['home_teamName']
nome_pais_visitante_valido=qtd_max_goals_validos.first()['away_teamName']
quantidade_gols_valido=qtd_max_goals_validos.first()['total_goals']
nome_torneio_valido=qtd_max_goals_validos.first()['tournamentName']

print(f'Partida com mais gols, tendo como base o campo total_goals, foi a partida entre {nome_pais_mandante} e {nome_pais_visitante} no torneio {nome_torneio} com {quantidade_gols} gols, isto considerando a base completa, já se consideramos somente os validos seria uma partida entre {nome_pais_mandante_valido} e {nome_pais_visitante_valido} no torneio {nome_torneio_valido} com {quantidade_gols_valido} gols')

### 9. Qual a maior goleada?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window as W

# 1. Ranqueia pela maior diferença absoluta de gols
df_rank = (
    df_transformado
        .withColumn("score_diff", F.expr("abs(home_scoreHome - away_scoreAway)"))
        .withColumn("rank_goleada", F.rank().over(W.partitionBy(F.lit(1)).orderBy(F.desc("score_diff"))))
)

# Filtra apenas o topo do ranking
df_top = df_rank.filter(F.col("rank_goleada") == 1)
df_top_val = df_rank.filter((F.col("rank_goleada") == 1) & (F.col("neutralTRUE") == False))

def extract_match_details(df_filtered):
    row = df_filtered.first()
    
    # Valida quem marcou mais gols no jogo
    if row["home_scoreHome"] >= row["away_scoreAway"]:
        winner, loser = row["home_teamName"], row["away_teamName"]
        w_score, l_score = row["home_scoreHome"], row["away_scoreAway"]
    else:
        winner, loser = row["away_teamName"], row["home_teamName"]
        w_score, l_score = row["away_scoreAway"], row["home_scoreHome"]
        
    return winner, loser, w_score, l_score, row["tournamentName"]

def get_winner(df_top, df_top_val):
    p_ganhador, p_perdedor, g_ganhador, g_perdedor, torneio = extract_match_details(df_top)
    p_ganhador_val, p_perdedor_val, g_ganhador_val, g_perdedor_val, torneio_val = extract_match_details(df_top_val)

    print(
        f"A partida com maior goleada foi a partida entre {p_ganhador} e {p_perdedor} "
        f"no torneio {torneio} com {g_ganhador} gols do {p_ganhador} e {g_perdedor} gols do {p_perdedor}. "
        f"Já considerando apenas registros válidos foi uma partida entre {p_ganhador_val} e {p_perdedor_val} "
        f"no torneio {torneio_val} com {g_ganhador_val} gols do {p_ganhador_val} e {g_perdedor_val} gols do {p_perdedor_val}."
    )

get_winner(df_top, df_top_val)

    

A partida com maior goleada foi a partida entre Australia e American Samoa no torneio FIFA World Cup qualification com 31 gols do Australia e 0 gols do American Samoa. Já considerando apenas registros válidos foi uma partida entre Australia e American Samoa no torneio FIFA World Cup qualification com 31 gols do Australia e 0 gols do American Samoa.


### 10. Quantos jogos ocorreram no Brasil?

In [0]:
qtd_jogos_totais_br= df_transformado.filter(df_transformado.countryCountry.isin('Brazil','Brasil')).count()
qtd_jogos_validos_br= df_transformado.filter((df_transformado.countryCountry.isin('Brazil','Brasil')) & (df_transformado.neutralTRUE==False)).count()
print(f"Quantidade de jogos totais no Brasil: {qtd_jogos_totais_br}")
print(f"Quantidade de jogos válidos no Brasil: {qtd_jogos_validos_br}")

Quantidade de jogos totais no Brasil: 529
Quantidade de jogos válidos no Brasil: 335
